# Unified ML Model Training - Build-a-Bear for ML
"
**One notebook to train them all.**
"
Configure everything in Cell 1, then Run All Cells.
"
## Features:
- Train ANY of 23 models (or all at once)
- Per-model timeframe selection (XGBoost@15min + LSTM@5min from same 1-min source)
- Per-model feature optimization (Optuna-based feature pruning)
- Heterogeneous ensembles with meta-learners
- Optuna hyperparameter optimization
- Cross-validation support
"
## Quick Start:
1. Edit Cell 1 parameters (MODELS list)
2. Run All Cells
3. Models saved to Google Drive
"
## Example Configurations:
"
```python
# Single model with feature optimization
MODELS = [{"name": "xgboost", "timeframe": "15min", "optimize_features": True}]
"
# Heterogeneous ensemble (different timeframes)
MODELS = [
    {"name": "xgboost", "timeframe": "15min", "optimize_features": True},
    {"name": "lstm", "timeframe": "5min", "optimize_features": True, "sequence_length": 60},
    {"name": "patchtst", "timeframe": "1min"},  # Raw OHLCV only
]
BUILD_ENSEMBLE = True
```

In [ ]:
# ============================================================================
# CONFIGURATION - EDIT THIS CELL
# ============================================================================

# Data Configuration
SYMBOL = "MES"                              # Contract: MES, MGC, ES, GC
HORIZONS = [20]                             # Label horizons: [5, 10, 15, 20]

# Model Selection (Build-a-Bear!)
# Each model can have its own timeframe and optimization settings
MODELS = [
    {"name": "xgboost", "timeframe": "15min", "optimize_features": True, "feature_opt_trials": 30},
    {"name": "lstm", "timeframe": "5min", "optimize_features": True, "sequence_length": 60},
]
# Simple string list also works (uses defaults):
# MODELS = ["xgboost", "lightgbm"]

# Available models:
# Tabular: xgboost, lightgbm, catboost, random_forest, logistic, svm
# Neural: lstm, gru, tcn, transformer, patchtst, itransformer, tft, nbeats, inceptiontime, resnet1d
# Ensemble: voting, stacking, blending
# Meta: ridge_meta, mlp_meta, calibrated_meta, xgboost_meta

# Ensemble Configuration
BUILD_ENSEMBLE = True                       # Build ensemble from trained models
ENSEMBLE_METHOD = "stacking"                # voting/stacking/blending
META_LEARNER = "ridge_meta"                 # For stacking: ridge_meta/mlp_meta/xgboost_meta

# Global Optimization Settings (applied to all models if not overridden)
GLOBAL_FEATURE_OPTIMIZATION = False         # Run Optuna feature selection for all models
GLOBAL_HYPERPARAM_OPTIMIZATION = False      # Run Optuna HPO for all models

# Cross-Validation
CROSS_VALIDATE = True                       # Run cross-validation
CV_SPLITS = 5                               # Number of CV folds

# Output
SAVE_TO_DRIVE = True                        # Save results to Google Drive
DRIVE_PATH = "/content/drive/MyDrive/ML_Factory_Results"  # Drive save location

## Setup Environment

In [ ]:
# Install dependencies
!pip install -q xgboost lightgbm catboost optuna pywavelets numba scikit-learn pandas numpy matplotlib seaborn pyyaml

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repo or set path to your Research directory
import sys
import os

# Option 1: If repo is in Google Drive
REPO_PATH = "/content/drive/MyDrive/Research"

# Option 2: Clone from GitHub (uncomment if needed)
# !git clone https://github.com/your-username/Research.git /content/Research
# REPO_PATH = "/content/Research"

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {REPO_PATH}")

## Load Configuration

In [ ]:
from pathlib import Path
from src.training import TrainingOrchestrator, ExperimentConfig, ModelConfig

print("Creating experiment configuration...")

config = ExperimentConfig(
    symbol=SYMBOL,
    horizons=HORIZONS,
    models=MODELS,  # Can be list of dicts or list of strings
    data_dir=Path("data/splits/scaled"),
    output_dir=Path("experiments/colab_runs"),
    cross_validate=CROSS_VALIDATE,
    cv_splits=CV_SPLITS,
    build_ensemble=BUILD_ENSEMBLE,
    ensemble_method=ENSEMBLE_METHOD,
    meta_learner=META_LEARNER,
    global_feature_optimization=GLOBAL_FEATURE_OPTIMIZATION,
    global_hyperparam_optimization=GLOBAL_HYPERPARAM_OPTIMIZATION,
)

print("\nConfiguration:")
print(f"  Symbol: {config.symbol}")
print(f"  Horizons: {config.horizons}")
print(f"  Models:")
for m in config.models:
    timeframe = m.timeframe or '(default 5min)'
    opt_features = 'opt-features' if m.optimize_features else 'baseline-features'
    print(f"    - {m.name:15s} @ {timeframe:10s} [{opt_features}]")
print(f"  Build ensemble: {config.build_ensemble}")
if config.build_ensemble:
    print(f"    Method: {config.ensemble_method}")
    print(f"    Meta-learner: {config.meta_learner}")

## Run Training Pipeline

In [ ]:
print("="*60)
print("STARTING UNIFIED TRAINING PIPELINE")
print("="*60)

orchestrator = TrainingOrchestrator(config)
results = orchestrator.run()

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

## Display Results

In [ ]:
orchestrator.display_results()

## Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')

for horizon_key, horizon_results in results.items():
    if not horizon_results:
        continue
    
    model_names = []
    f1_scores = []
    accuracies = []
    
    for model_name, model_data in horizon_results.items():
        if isinstance(model_data, dict) and 'results' in model_data:
            model_names.append(model_name)
            f1_scores.append(model_data['results']['evaluation_metrics']['val_f1'])
            accuracies.append(model_data['results']['evaluation_metrics']['val_accuracy'])
    
    if model_names:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        axes[0].barh(model_names, f1_scores, color='steelblue')
        axes[0].set_xlabel('Validation F1 Score')
        axes[0].set_title(f'{horizon_key} - F1 Scores')
        axes[0].set_xlim([0, 1])
        
        axes[1].barh(model_names, accuracies, color='coral')
        axes[1].set_xlabel('Validation Accuracy')
        axes[1].set_title(f'{horizon_key} - Accuracy')
        axes[1].set_xlim([0, 1])
        
        plt.tight_layout()
        plt.show()
        
        save_path = orchestrator.output_dir / f"results_{horizon_key}.png"
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved: {save_path}")

## Save to Google Drive (Optional)

In [ ]:
if SAVE_TO_DRIVE:
    import shutil
    from pathlib import Path
    
    drive_save_path = Path(DRIVE_PATH)
    drive_save_path.mkdir(parents=True, exist_ok=True)
    
    run_id = orchestrator.output_dir.name
    final_save_path = drive_save_path / run_id
    
    if final_save_path.exists():
        shutil.rmtree(final_save_path)
    
    shutil.copytree(orchestrator.output_dir, final_save_path)
    
    print(f"✅ Results saved to Google Drive: {final_save_path}")
    print(f"\nSaved files:")
    for file in final_save_path.rglob('*'):
        if file.is_file():
            print(f"  - {file.relative_to(final_save_path)}")
else:
    print("Skipping Google Drive save (SAVE_TO_DRIVE=False)")
    print(f"Results available at: {orchestrator.output_dir}")

## Summary & Next Steps

---
# 🚀 NEW: Unified MLPipeline Interface (2026-01-16)

**The examples above use the `TrainingOrchestrator` (training-only interface).**

**NEW in 2026-01-16:** The `MLPipeline` class provides a **unified interface** for the **ENTIRE pipeline** (data + training + evaluation) in ONE call.

## Why MLPipeline?

| Old Approach | New Approach (MLPipeline) |
|--------------|---------------------------|
| Run `pipeline run` (data) | ✅ ONE call: `MLPipeline.run()` |
| Run `python scripts/train_model.py` (training) | |
| Run `python scripts/run_cv.py` (evaluation) | |
| = 3 separate commands | = Full pipeline in ONE call |

## Features:
- **Unified config** - One `MLConfig` for everything
- **Checkpoint/resume** - Resume from any point if interrupted
- **Phase control** - Run all phases or specific phases
- **CLI + Python API** - Use from command line or notebooks
- **All 23 models** - Same model support as TrainingOrchestrator

---

## Example 1: Full Pipeline (Data → Training → Evaluation)

In [ ]:
# NEW: MLPipeline interface (full pipeline in ONE call)
from src.ml_pipeline import MLPipeline, MLConfig

# Create unified config
config = MLConfig(
    # Data configuration
    symbol="MES",
    horizons=[20],
    start_date="2020-01-01",
    end_date="2024-12-31",
    
    # Model configuration (same as TrainingOrchestrator)
    models=["xgboost", "lstm"],
    training_mode="standard",
    build_ensemble=True,
    
    # Evaluation configuration
    evaluation_method="cv",
    cv_splits=5,
)

# Run FULL pipeline (data + training + evaluation)
pipeline = MLPipeline(config)
# results = pipeline.run()  # Uncomment when data is available

print("✅ Full pipeline configured!")
print(f"   Data: {config.symbol} ({config.start_date} to {config.end_date})")
print(f"   Models: {config.models}")
print(f"   Ensemble: {config.build_ensemble}")
print(f"   Evaluation: {config.evaluation_method} ({config.cv_splits} splits)")

## Example 2: Phase-by-Phase Execution

In [ ]:
# Run phases separately for fine-grained control
from src.ml_pipeline import MLPipeline, MLConfig

config = MLConfig(
    symbol="MES",
    horizons=[20],
    models=["xgboost", "lightgbm"],
)

pipeline = MLPipeline(config)

# Phase 1: Data pipeline only
# data_results = pipeline.run_data()
print("Phase 1: Data pipeline (ingest → features → labels → scaling)")

# Phase 2: Training only (uses data from Phase 1)
# training_results = pipeline.run_training()
print("Phase 2: Training (base models + ensemble)")

# Phase 3: Evaluation only
# evaluation_results = pipeline.run_evaluation()
print("Phase 3: Evaluation (cross-validation, metrics)")

print("\n✅ Phase-by-phase execution allows checkpoint/resume!")

## Example 3: Checkpoint and Resume

In [ ]:
# Resume from checkpoint if pipeline was interrupted
from src.ml_pipeline import MLPipeline, MLConfig
from pathlib import Path

# Option 1: Resume from run_id
# pipeline = MLPipeline.from_checkpoint(run_id="20260116_120000")
# results = pipeline.resume()

# Option 2: Resume from state file
# state_file = Path("experiments/runs/20260116_120000/pipeline_state.json")
# pipeline = MLPipeline.from_checkpoint(state_file=state_file)
# results = pipeline.resume()

print("✅ Resume feature allows recovery from any failure point!")
print("   Checkpoints saved after: data_complete, training_complete, evaluation_complete")

## Example 4: Advanced Configuration (Heterogeneous Ensemble)

In [ ]:
# Heterogeneous ensemble with different timeframes
from src.ml_pipeline import MLConfig, ModelConfig

config = MLConfig(
    symbol="MES",
    horizons=[20],
    
    # Per-model timeframe configuration
    models=[
        ModelConfig(
            name="xgboost",
            timeframe="15min",  # CatBoost trains on 15min
            optimize_features=True,
        ),
        ModelConfig(
            name="lstm",
            timeframe="5min",   # LSTM trains on 5min
            sequence_length=60,
            optimize_features=True,
        ),
        ModelConfig(
            name="patchtst",
            timeframe="1min",   # PatchTST uses raw 1min OHLCV
        ),
    ],
    
    # Ensemble configuration
    build_ensemble=True,
    meta_learner="ridge_meta",
    
    # Training mode
    training_mode="standard",  # or "walk_forward", "regime_aware", "meta_labeling"
)

print("✅ Heterogeneous ensemble configured!")
print("   Base models:")
for m in config.models:
    print(f"     - {m.name:12s} @ {m.timeframe or '5min (default)'}")
print(f"   Meta-learner: {config.meta_learner}")
print("\n📝 All models train on SAME 1-min canonical OHLCV source!")
print("   Different timeframes are resampled from 1-min data.")

## Example 5: CLI Usage (Alternative to Notebook)

In [ ]:
# You can also use the CLI for the same functionality
print("Command-line interface examples:\n")

print("1. Full pipeline:")
print("   ml run --symbol MES --models xgboost lstm --build-ensemble\n")

print("2. Data only:")
print("   ml data --symbol MES --horizons 20\n")

print("3. Training only:")
print("   ml train --models xgboost --training-mode standard\n")

print("4. Resume from checkpoint:")
print("   ml resume --run-id 20260116_120000\n")

print("5. Check status:")
print("   ml status --run-id 20260116_120000\n")

print("✅ All CLI commands support YAML config files!")
print("   ml run --config experiments/my_config.yaml")

## Comparison: TrainingOrchestrator vs MLPipeline

| Feature | TrainingOrchestrator | MLPipeline |
|---------|---------------------|------------|
| **Scope** | Training only | Full pipeline (data + training + eval) |
| **Data pipeline** | ❌ Manual (run `pipeline run` first) | ✅ Automatic |
| **Training** | ✅ Supported | ✅ Supported (delegates to TrainingOrchestrator) |
| **Evaluation** | ❌ Manual (run separate scripts) | ✅ Automatic |
| **Checkpoint/Resume** | ❌ No | ✅ Yes |
| **Phase control** | ❌ All-or-nothing | ✅ Run specific phases |
| **CLI interface** | ❌ Via `scripts/train_model.py` | ✅ Via `ml` command |
| **Config** | ExperimentConfig | MLConfig (superset) |
| **Use case** | Quick training experiments | Full production pipeline |

### When to use which?

**Use `TrainingOrchestrator`** (cells above) when:
- You already ran the data pipeline manually
- You only need training (no evaluation)
- Quick experimentation in notebooks

**Use `MLPipeline`** (this section) when:
- You want the full pipeline (data → training → eval)
- You need checkpoint/resume functionality
- You want reproducible, end-to-end runs
- You're building production workflows

---

## Next Steps

1. **Try TrainingOrchestrator first** (cells above) - requires pre-processed data
2. **Then try MLPipeline** (this section) - runs entire pipeline
3. **Compare results** - same models, different interfaces
4. **Use MLPipeline for production** - checkpoint/resume, full control

**Documentation:**
- `docs/implementation/UNIFIED_PIPELINE_ARCHITECTURE.md` - Complete architecture
- `docs/UNIFIED_PIPELINE_ROADMAP.md` - Implementation roadmap
- `examples/unified_pipeline_basic.py` - Python API examples

In [ ]:
print("="*60)
print("EXPERIMENT SUMMARY")
print("="*60)

print(f"\nExperiment: {config['experiment']['name']}")
print(f"Symbol: {config['data']['symbol']}")
print(f"Horizons: {config['data']['horizons']}")
print(f"Models trained: {len(config['models']['model_list'])}")

print("\nBest Model per Horizon:")
for horizon_key, horizon_results in results.items():
    if not horizon_results:
        continue
    
    best_model = None
    best_f1 = 0
    
    for model_name, model_data in horizon_results.items():
        if isinstance(model_data, dict) and 'results' in model_data:
            f1 = model_data['results']['evaluation_metrics']['val_f1']
            if f1 > best_f1:
                best_f1 = f1
                best_model = model_name
    
    if best_model:
        print(f"  {horizon_key}: {best_model} (F1={best_f1:.4f})")

print(f"\n{'='*60}")
print("NEXT STEPS:")
print("="*60)
print("1. Review results in cells above")
print("2. Check saved models and plots")
if SAVE_TO_DRIVE:
    print(f"3. Results in Google Drive: {DRIVE_PATH}")
print("4. Try different model combinations or timeframes in Cell 1 and re-run")
print("5. Enable optimize_features=True per model for better performance")
print("6. Try heterogeneous ensembles: XGBoost@15min + LSTM@5min + PatchTST@1min")